# Lab 2.1 – Datenqualität prüfen

Jede Zelle prüft eine Sache an der Rohdatei. Zellen der Reihe nach mit **Umschalt+Enter** ausführen. Das Notebook ändert die Daten nicht; am Ende steht eine Problemtabelle.

In [ ]:
import pandas as pd

ORDNER = "project/playground/"   # hier landet die Problemtabelle

df = pd.read_csv("data/nordwind_umsatz.csv")
df.head()

## Prüfung 1: Sind alle Pflichtspalten da?

In [ ]:
PFLICHT = ["beleg_nr", "datum", "jahr", "kunde_id", "region", "produktgruppe",
           "kanal", "menge", "umsatz", "kosten"]
fehlende_spalten = [s for s in PFLICHT if s not in df.columns]
print("Spalten:", list(df.columns))
print("Fehlende Pflichtspalten:", fehlende_spalten)

## Prüfung 2: Typ und Füllstand jeder Spalte

In [ ]:
df.info()

## Prüfung 3: Fehlende Werte je Spalte

In [ ]:
fehlend = df.isna().sum()
fehlend

## Prüfung 4: Dubletten (Zeilen, die genau so schon einmal vorkommen)

In [ ]:
dubletten = df.duplicated().sum()
print("Dubletten:", dubletten)

## Prüfung 5: Datumsformate

Der Onlineshop liefert `TT.MM.JJJJ` (mit Punkt), der Außendienst `JJJJ-MM-TT`.

In [ ]:
datum_deutsch = df["datum"].str.contains(".", regex=False).sum()
print("Datum mit Punkt (TT.MM.JJJJ):", datum_deutsch)
print("Datum mit Bindestrich (JJJJ-MM-TT):", len(df) - datum_deutsch)

## Prüfung 6: Ausreißer beim Umsatz

`describe()` zeigt Anzahl, Mittelwert, Streuung (`std`), Minimum, Quartile und Maximum.

In [ ]:
print(df["umsatz"].describe().round(2))
ausreisser = df[df["umsatz"] > 5000]   # mehr als 5.000 € in einer Zeile
ausreisser

## Prüfung 7: Schreibweisen der Regionen – Aufgabe 1

Der Code für diese Zelle kommt von Copilot (Prompt in der Übung, Aufgabe 2). Ergebnis: eine Variable `regionen` mit jeder Schreibweise und ihrer Zeilenzahl.

In [ ]:
# >>> AUFGABE 1: Schreibweisen der Spalte region zählen – Prompt siehe Übung, Aufgabe 2
# <<< ENDE AUFGABE 1

In [ ]:
# Prüfzelle – nicht ändern, nur ausführen
if "regionen" not in globals():
    print("nicht OK – Variable regionen fehlt: Code aus Aufgabe 1 einfügen und die Zelle darüber ausführen.")
elif not isinstance(regionen, pd.Series) or regionen.sum() != len(df):
    print("nicht OK – regionen soll je Schreibweise die Zeilenzahl enthalten (value_counts), über alle Zeilen.")
elif len(regionen) != df["region"].nunique():
    print(f"nicht OK – {len(regionen)} Schreibweisen, erwartet {df['region'].nunique()}: Leerzeichen und Groß-/Kleinschreibung zählen mit, nichts vorher bereinigen.")
else:
    print(f"OK – {len(regionen)} Schreibweisen für 4 Regionen gefunden.")

## Problemtabelle

Fasst alle Prüfungen zusammen. Die Spalte `Vorschlag` ist die Grundlage für die Bereinigung in Lab 2.2.

In [ ]:
anzahl_regionen = len(regionen) if "regionen" in globals() else "fehlt – Aufgabe 1"
probleme = pd.DataFrame([
    ["Dubletten (Zeile doppelt geliefert)", "alle", dubletten, "entfernen"],
    ["Kosten fehlen", "kosten", fehlend["kosten"], "markieren, nicht löschen"],
    ["Datum als TT.MM.JJJJ statt JJJJ-MM-TT", "datum", datum_deutsch, "beide Formate lesen"],
    ["Schreibweisen für 4 Regionen", "region", anzahl_regionen, "vereinheitlichen"],
    ["Umsatz über 5.000 € in einer Zeile", "umsatz", len(ausreisser), "fachlich prüfen"],
], columns=["Problem", "Spalte", "Anzahl", "Vorschlag"])
probleme.to_csv(ORDNER + "probleme.csv", index=False)
print("Gespeichert:", ORDNER + "probleme.csv")
probleme